# The V-cycle on a real slice: what each grid level holds

A trained MGLPDS net, run on one real val slice, and the image it holds at **every
grid level** on the way back up the V-cycle (the decoder side) -- next to what a
U-Net decoder (E2E-VarNet's) holds at the same scales.

**The argument.** In MGLPDS every level carries a primal *image* $x_\ell$ that solves
a coarse version of the same reconstruction problem (FAS: the coarse variable is the
full restricted solution, not an error), so it can be scored directly against the
ground truth restricted by the net's own transfer, $R^\ell(\mathrm{gt})$. A U-Net
decoder carries 18--144 feature *channels* per scale with no defined relation to an
image; showing them at all needs a summary.

The engine is `scripts/level_figure.py` (also a CLI). The measurement is exactly
`eval_per_volume`'s for the slice -- loader, noise seed, online ESPIRiT operator,
sigma -- so the finest level reproduces `results/main/slices.csv`.

**Two ways to run**
* **Live** (cluster, GPU node, data + checkpoints): set `RUN`, `COMPARE`, `VOLUME`,
  `SLICE`. You get the arrays *and* the live objects (model, `LevelTrace`, the sparse
  codes) for section 6.
* **From a saved trace** (anywhere, numpy + matplotlib only): set `NPZ` to a file
  `save_npz` / the CLI wrote, e.g. after copying `figures/out/levels/` down.

In [ ]:
import pathlib, sys

_root = pathlib.Path.cwd()
if not (_root / "models").is_dir():
    _root = _root.parent
sys.path.insert(0, str(_root))
sys.path.insert(0, str(_root / "scripts"))

import numpy as np
import matplotlib.pyplot as plt

import level_figure as lf
from visualization.image import subplot_images

# ---- EITHER a live run (cluster) ---------------------------------------------
RUN     = "trained_nets/mg_recon/knee/mg169v6_R8"
COMPARE = "trained_nets/mg_recon/knee/varnetmaps_R8"    # None: no U-Net comparison
VOLUME  = "file1000735"     # any unique part of the name (rank_slices' filter works)
SLICE   = 15                # SOURCE slice number, as the viewer and slices.csv
SIGMA   = None              # None: the run's val_noise_std (0.015)
VAL_KSPACE = "measurement_awgn"   # as results/main; "simulated" for that tree
# ---- OR a saved trace (anywhere) ---------------------------------------------
NPZ     = None              # e.g. "figures/out/levels/knee_1000735_15.npz"
SAVE_TO = "figures/out/levels"    # live mode saves the trace here (None: don't)
FLIP    = None              # None: 'ud' for knee (as the viewer), '' for brain

In [ ]:
if NPZ:
    z, meta = lf.load_npz(_root / NPZ if not pathlib.Path(NPZ).is_absolute() else NPZ)
    live = None
else:
    import os
    os.chdir(_root)                      # run dirs and the data roots are repo-relative
    z, meta, live = lf.trace_slice(RUN, VOLUME, SLICE, compare=COMPARE, sigma=SIGMA,
                                   val_kspace_type=VAL_KSPACE)
    if SAVE_TO:
        print("saved", lf.save_npz(z, meta, SAVE_TO))

FLIP = lf.default_flip(meta) if FLIP is None else FLIP
F = lambda a: lf._flip(a, FLIP) if FLIP else a          # display orientation only

print(f"{meta['run']}  M={meta['M']}  K={meta['K']}  {meta['n_params']:,} params   "
      f"{meta['volume']} slice {meta['slice']}  R={meta['R']}  sigma={meta['sigma']:g}")
print(f"DC recovered {complex(*meta['dc']):.4g}, misfit {meta['scale_misfit']:.1e} "
      f"(should be ~1e-7: the finest level + DC IS the output)")
for l, shape, errs in lf.level_table(z, meta):
    print(f"  level {l}  {shape[0]:>4}x{shape[1]:<4}  NRMSE vs R^l(gt) by V-cycle: "
          + "  ".join(f"{100 * e:5.1f}%" for e in errs))

## 1. The decoder side

One column per level, coarsest $\to$ finest, after the last V-cycle has come back up
through it. Rows: $x_\ell$; $R^\ell(\mathrm{gt})$; the residual (windowed at
vmax / `resid_gain`, as the viewer); and $|\Delta x|$, what the level below handed up
(on the grid it lives on; the finest level has none). Every panel is at its own grid
size -- nothing is resampled. `outer=` picks another V-cycle (0-based, -1 = last).

In [ ]:
fig = lf.fig_levels(z, meta, outer=-1, flip=FLIP)
plt.show()

## 2. Across the outer V-cycles

Each level after each of the outer cycles; panel titles are NRMSE vs $R^\ell(\mathrm{gt})$.
The coarse levels settle first -- the low frequencies are what a coarse grid can fix --
and the fine level keeps sharpening.

In [ ]:
fig = lf.fig_outers(z, meta, flip=FLIP)
plt.show()

# the same numbers as curves
fig, ax = plt.subplots(figsize=(5, 3.4))
for l, shape, errs in lf.level_table(z, meta):
    ax.plot(np.arange(1, len(errs) + 1), [100 * e for e in errs], marker="o",
            label=f"level {l} ({shape[0]}x{shape[1]})")
ax.set_xlabel("outer V-cycle"); ax.set_ylabel("NRMSE vs R^l(gt)  (%)")
ax.legend(frameon=False); ax.grid(alpha=.3)
plt.show()

## 3. The U-Net decoder at its scales

The compared VarNet's last cascade: its U-Net's decoder blocks, coarse $\to$ fine.
Channel RMS and the first principal component (and how much of the channel variance
it explains) -- a summary is the only way to look at them at all.

In [ ]:
fig = lf.fig_unet(z, meta, flip=FLIP)
if fig is None:
    print("no comparison in this trace (COMPARE=None)")
else:
    plt.show()

## 4. Do it yourself

Everything is in `z` (numpy) and `meta`. Accessors below; `F()` applies the display
flip. `subplot_images` (visualization/image.py) shares one window across panels
unless told otherwise -- pass `share_window=False` for panels on different scales.

In [ ]:
LEVELS, OUTERS = meta["levels"], meta["outers"]
X  = lambda l, o=-1: z[f"mg/x/o{OUTERS[o]}/l{l}"]                 # complex, image units
GT = lambda l: z[f"mg/gt/l{l}"]                                   # R^l(gt)
D  = lambda l, o=-1: z.get(f"mg/delta/o{OUTERS[o]}/l{l}")         # handed up (l >= 1)
ORGAN = lambda l: z[f"mg/organ/l{l}"]                             # R^l(organ mask)
FEAT = sorted((k for k in z if k.startswith("vn/feat/")), key=lambda k: int(k.rsplit("/", 1)[1]))

print("levels", LEVELS, "outers", OUTERS, "| U-Net scales", [z[k].shape for k in FEAT])

# example: the final reconstruction next to every coarse level, each on its own window
subplot_images([F(np.abs(X(l))) for l in sorted(LEVELS, reverse=True)] + [F(np.abs(GT(0)))],
               titles=[f"level {l}" for l in sorted(LEVELS, reverse=True)] + ["ground truth"],
               share_window=False)

In [ ]:
# NRMSE inside the organ (the region the metrics score) vs the whole frame, per level
for l in sorted(LEVELS, reverse=True):
    m = ORGAN(l) > 0.5
    x, g = np.abs(X(l)), np.abs(GT(l))
    inside = np.linalg.norm((x - g)[m]) / np.linalg.norm(g[m])
    print(f"level {l}: whole frame {100 * lf.nrmse(X(l), GT(l)):5.1f}%   organ {100 * inside:5.1f}%")

In [ ]:
# browse individual U-Net channels at one scale (0 = coarsest decoder block)
if FEAT:
    K = 0
    f = z[FEAT[K]].astype(np.float32)
    top = np.argsort(-(f ** 2).mean((1, 2)))[:16]            # the 16 most energetic
    subplot_images([F(f[c]) for c in top], titles=[f"ch {c}" for c in top],
                   ncols=8, cmap="RdBu_r", symmetric=True, share_window=False,
                   suptitle=f"U-Net decoder block {K}: {f.shape[0]} channels at {f.shape[1]}x{f.shape[2]}")

## 5. Zoom: the same region at every level

A box given in FINEST-grid pixels (displayed orientation), mapped onto each coarser
grid by the factor 2 per level -- how much detail each level actually carries there.

In [ ]:
BOX = None    # (row, col, height, width) on the finest displayed grid; None: centre quarter
H0, W0 = X(0).shape
r, c, h, w = BOX or (H0 // 2 - H0 // 8, W0 // 2 - W0 // 8, H0 // 4, W0 // 4)
ims, titles = [], []
for l in sorted(LEVELS, reverse=True):
    s = 2 ** l
    for name, a in (("x", X(l)), ("gt", GT(l))):
        A = F(np.abs(a))
        ims.append(A[r // s:(r + h) // s, c // s:(c + w) // s]); titles.append(f"{name} level {l}")
subplot_images(ims, titles=titles, ncols=2, share_window=True,
               suptitle=f"rows {r}:{r + h}, cols {c}:{c + w} of the finest grid")

## 6. Live only: the sparse codes, the FAS corrections, the model

`live["mg"]["trace"]` is the full `LevelTrace` (models/level_trace.py): per
(outer, level) the incoming / outgoing iterates `x`, `z`, `x_out`, `z_out`, what the
level below receives (`x_c`, `z_c`), the FAS corrections (`pi_x`, `pi_z`) and
`coarse_delta_*`. These are on the NETWORK's scale (DC removed), and under the planar
state they are real with [re; im] stacked on the channel axis -- `as_complex` undoes it.

`z` is the dual / code variable: M channels per level, on the LATENT grid -- the
image grid divided by the conv stride `s` (2), so half each level's image size.
Unlike a U-Net's channels each one is tied to a dictionary atom
(`visualization.filters.effective_atoms(model)`).

In [ ]:
if live is None:
    print("loaded from NPZ: no live objects -- run in live mode for this section")
else:
    import torch
    tr, model = live["mg"]["trace"], live["mg"]["model"]
    print(tr.summary())

    def as_complex(t):
        # the planar state stores [re; im] stacked on channels
        if t is None or t.is_complex():
            return t
        h = t.shape[1] // 2
        return torch.complex(t[:, :h], t[:, h:])

    # code energy per channel at each level, last V-cycle
    for l in sorted(LEVELS, reverse=True):
        zc = as_complex(tr.get(-1, l, "z_out"))
        if zc is None:
            continue
        e = zc.abs().pow(2).mean((0, 2, 3)).cpu().numpy()
        print(f"level {l}: z {tuple(zc.shape)}  active channels (>1% of max energy): "
              f"{int((e > 0.01 * e.max()).sum())}/{e.size}")

In [ ]:
# the |z| maps of the most energetic code channels at one level -- each channel is an atom's coefficient map
if live is not None:
    L = max(LEVELS)                      # try 0 for the finest
    zc = as_complex(tr.get(-1, L, "z_out"))[0].abs().cpu().numpy()
    top = np.argsort(-(zc ** 2).mean((1, 2)))[:16]
    subplot_images([F(zc[m]) for m in top], titles=[f"atom {m}" for m in top], ncols=8,
                   share_window=False, suptitle=f"|z| at level {L}: the 16 most active atoms")